# ML-05 — CTR / Engagement Opportunity Scoring: Model vs Baseline

**Lane:** CTR / Engagement Opportunity Scoring

**Question:** Can a small, interpretable learned model prioritize pages that show a future CTR drop better than the transparent Week-4 baseline?

This notebook uses only information available at the decision month. It compares a Decision Tree with the Week-4-style position/CTR/volume baseline on the same time-aware evaluation set and the same Precision@50 metric.

**Important:** this is decision support, not proof of Google's ranking algorithm or proof that a particular content change causes improvement.

## 1. Method choice and why

I use a **Decision Tree classifier** because the lane is a ranking/prioritization problem with nonlinear interactions between visibility, CTR and position. A shallow tree is also easy to inspect and less opaque than a complex ensemble.

The model predicts whether a page's next-month CTR is lower than its current-month CTR. The prediction is converted into a ranked queue using the model's probability of a future CTR drop.

The baseline is intentionally simple: it prioritizes pages with meaningful impressions whose current CTR is low relative to the typical CTR for their current position bucket, with impression volume used as a practical priority multiplier.

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN secret not found in Colab.")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

print("Warehouse connection ready.")

Warehouse connection ready.


## 2. Split design

I use a **time-aware split** because the intended use is forward-looking. Features are calculated from month *t* and the label is calculated from month *t+1*.

- March 2026 → training
- April 2026 → validation
- May 2026 → final test
- June 2026 is used only as the future outcome for May and is never used as a feature.

No future-window field is included in the model features.

In [ ]:
months = con.sql(f'''
SELECT month, COUNT(*) AS rows
FROM {FACT}
GROUP BY month
ORDER BY month
''').df()

display(months)

In [2]:
monthly = con.sql(f'''
SELECT
    client_hash_id,
    content_hash_id,
    month,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
        ELSE 0.0
    END AS ctr,
    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS avg_position,
    SUM(ga4_sessions) AS ga4_sessions,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
    SUM(scroll_events) AS scroll_events,
    MAX(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available,
    MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available
FROM {FACT}
WHERE month IN ('2026-03','2026-04','2026-05','2026-06')
GROUP BY client_hash_id, content_hash_id, month
''').df()

monthly["engagement_rate"] = np.where(
    monthly["ga4_sessions"] > 0,
    monthly["ga4_engaged_sessions"] / monthly["ga4_sessions"],
    np.nan
)

print("Monthly rows:", len(monthly))
display(monthly.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Monthly rows: 1491967


,client_hash_id,content_hash_id,month,impressions,clicks,ctr,avg_position,ga4_sessions,ga4_engaged_sessions,scroll_events,gsc_available,ga4_available,engagement_rate
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03,1140.0,2.0,0.001754,4.450877,0.0,0.0,0.0,1,0,NaN
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03,57.0,0.0,0.000000,2.298246,0.0,0.0,0.0,1,0,NaN
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03,6523.0,7.0,0.001073,6.893301,1.0,0.0,0.0,1,1,0.0
3,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03,5630.0,6.0,0.001066,6.535346,3.0,0.0,0.0,1,1,0.0
4,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03,48.0,0.0,0.000000,18.145833,0.0,0.0,0.0,1,0,NaN


In [3]:
def make_pair(current_month, future_month):
    cur = monthly[monthly["month"] == current_month].copy()
    fut = monthly[monthly["month"] == future_month][
        ["client_hash_id", "content_hash_id", "ctr"]
    ].rename(columns={"ctr": "future_ctr"})

    out = cur.merge(
        fut,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )

    out = out[out["impressions"] >= 50].copy()

    # Target/proxy: next-month CTR is lower than current-month CTR.
    out["target"] = (out["future_ctr"] < out["ctr"]).astype(int)

    out["position_bucket"] = pd.cut(
        out["avg_position"],
        bins=[-np.inf, 3, 10, 20, 50, np.inf],
        labels=["1-3", "4-10", "11-20", "21-50", "51+"]
    )
    return out

train = make_pair("2026-03", "2026-04")
valid = make_pair("2026-04", "2026-05")
test = make_pair("2026-05", "2026-06")

print("Train:", train.shape, "positive rate:", round(train["target"].mean(), 4))
print("Valid:", valid.shape, "positive rate:", round(valid["target"].mean(), 4))
print("Test :", test.shape,  "positive rate:", round(test["target"].mean(), 4))

Train: (116114, 16) positive rate: 0.4016
Valid: (125758, 16) positive rate: 0.277
Test : (133719, 16) positive rate: 0.321


### Baseline construction

The Week-4 baseline used the same lane logic: CTR must be interpreted relative to search position, and pages with more impressions have more practical review value.

For each evaluation month, the baseline ranks pages by:

**opportunity = position-relative CTR gap × log(1 + impressions)**

This is a transparent rule, not a learned model.

In [4]:
def add_baseline_score(df):
    x = df.copy()
    bucket_median = x.groupby("position_bucket", observed=True)["ctr"].transform("median")
    x["position_ctr_gap"] = (bucket_median - x["ctr"]).clip(lower=0)
    x["baseline_score"] = x["position_ctr_gap"] * np.log1p(x["impressions"])
    return x

train = add_baseline_score(train)
valid = add_baseline_score(valid)
test = add_baseline_score(test)

display(test[[
    "client_hash_id", "content_hash_id", "impressions", "ctr",
    "avg_position", "position_ctr_gap", "baseline_score", "target"
]].sort_values("baseline_score", ascending=False).head(10))

,client_hash_id,content_hash_id,impressions,ctr,avg_position,position_ctr_gap,baseline_score,target
228379,client_157ffe4d4a595515,content_9648c4d1595a0794,219982.0,0.000236,1.946218,0.004942,0.060787,0
68220,client_23a62021009f63c4,content_bddfdd871aa09fbe,112353.0,0.000036,1.174094,0.005142,0.059802,0
153458,client_8ddc46da5414ffd8,content_d0acf7062bc6b257,129493.0,0.000556,2.227178,0.004622,0.054406,1
153577,client_8ddc46da5414ffd8,content_576d3cd50e6243c8,39579.0,0.000126,2.028374,0.005052,0.053477,1
347472,client_8ddc46da5414ffd8,content_943dc881428182b8,353426.0,0.001256,2.489848,0.003922,0.050101,0
153509,client_8ddc46da5414ffd8,content_f7d3a8b736eb7688,43658.0,0.000527,2.082184,0.004651,0.049693,0
225225,client_23a62021009f63c4,content_dd159d26bd77a5ad,14468.0,0.000000,2.343309,0.005178,0.049603,0
30117,client_23a62021009f63c4,content_6982fdcd6a6b28f8,19659.0,0.000203,2.679943,0.004974,0.049179,0
220216,client_08a6a72ff48e62c0,content_13ef916471e84dd8,14584.0,0.000069,2.158118,0.005109,0.048987,1
299665,client_e547b89c05043229,content_6c2da5d67fa888cc,14230.0,0.000141,1.555657,0.005037,0.048173,0


## 3. Train + compare vs my baseline

The model uses five decision-time features:

1. impressions
2. current CTR
3. average position
4. GA4 engagement rate
5. scroll events

These are observable before the future outcome. Future CTR and the target are used only for evaluation.

In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score
from sklearn.inspection import permutation_importance

FEATURES = [
    "impressions",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_events"
]

def prepare_X(df):
    x = df[FEATURES].replace([np.inf, -np.inf], np.nan).copy()
    return x.fillna(x.median(numeric_only=True))

X_train = prepare_X(train)
y_train = train["target"]

X_valid = prepare_X(valid)
y_valid = valid["target"]

X_test = prepare_X(test)
y_test = test["target"]

# Tune only on the validation month.
validation_rows = []
for depth in [2, 3, 4, 5, 6]:
    candidate = DecisionTreeClassifier(
        max_depth=depth,
        min_samples_leaf=50,
        random_state=42
    )
    candidate.fit(X_train, y_train)
    valid_score = candidate.predict_proba(X_valid)[:, 1]
    validation_rows.append({
        "max_depth": depth,
        "validation_Precision@50": precision_at_k(
            y_valid, valid_score, 50
        )
    })

validation_results = pd.DataFrame(validation_rows)
display(validation_results)

best_depth = int(
    validation_results.sort_values(
        "validation_Precision@50", ascending=False
    ).iloc[0]["max_depth"]
)

# Refit using train + validation after selecting the depth.
train_valid = pd.concat([train, valid], ignore_index=True)
X_train_valid = prepare_X(train_valid)
y_train_valid = train_valid["target"]

model = DecisionTreeClassifier(
    max_depth=best_depth,
    min_samples_leaf=50,
    random_state=42
)
model.fit(X_train_valid, y_train_valid)

test["model_score"] = model.predict_proba(X_test)[:, 1]

print("Selected max_depth:", best_depth)


NameError: name 'precision_at_k' is not defined

In [ ]:
def precision_at_k(y, score, k=50):
    order = np.argsort(-np.asarray(score))
    top = np.asarray(y)[order[:min(k, len(order))]]
    return float(top.mean()) if len(top) else np.nan

results = pd.DataFrame([
    {
        "method": "Week-4 transparent baseline",
        "Precision@50": precision_at_k(
            test["target"], test["baseline_score"], 50
        ),
        "Average Precision": average_precision_score(
            test["target"], test["baseline_score"]
        )
    },
    {
        "method": "Decision Tree",
        "Precision@50": precision_at_k(
            test["target"], test["model_score"], 50
        ),
        "Average Precision": average_precision_score(
            test["target"], test["model_score"]
        )
    }
])

display(results)

### Interpretation

The model is only useful if it improves the actual review queue rather than merely producing a more complicated score. Precision@50 is the primary metric because the downstream action is a limited top-of-queue review.

If the tree does not beat the baseline, that is still a valid result: the transparent rule may already capture most of the useful signal available from these features.

In [ ]:
perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring="average_precision"
)

importance = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

display(importance)

## 4. Errors and interpretation

The model's false positives are pages placed near the top whose next-month CTR did not fall. False negatives are pages whose CTR did fall but which the model did not prioritize.

These errors matter because a reviewer has finite capacity. A false positive spends review time; a false negative can leave a useful opportunity undiscovered.

The model should therefore be treated as a **prioritization aid**, not as an automatic recommendation to change titles, metadata or content.

In [ ]:
test_review = test[[
    "client_hash_id", "content_hash_id",
    "impressions", "ctr", "future_ctr", "avg_position",
    "baseline_score", "model_score", "target"
]].copy()

test_review["model_rank"] = test_review["model_score"].rank(
    ascending=False, method="first"
)
test_review["baseline_rank"] = test_review["baseline_score"].rank(
    ascending=False, method="first"
)

top50_cutoff = test_review["model_score"].nlargest(
    min(50, len(test_review))
).min()

test_review["model_error"] = np.select(
    [
        (test_review["model_score"] >= top50_cutoff) & (test_review["target"] == 0),
        (test_review["model_score"] < top50_cutoff) & (test_review["target"] == 1)
    ],
    ["false_positive_in_top50", "missed_positive"],
    default="other"
)

display(test_review.sort_values("model_rank").head(10))

## What the model can and cannot claim

**Can claim:** this experiment measures whether a small Decision Tree can prioritize the defined future-CTR-drop proxy better than the transparent baseline under this time-aware split.

**Cannot claim:** that the model discovers Google's ranking factors, that a low-CTR page definitely needs a specific edit, or that changing a page will causally improve search performance.

A future rerun should test stability across additional months and compare the queue at the same reviewer capacity.

## 5. Self-check

- Method chosen for the CTR / Engagement Opportunity Scoring lane.
- Time-aware train/validation/test design.
- Model compared with the Week-4-style baseline on the same test rows.
- Precision@50 reported for both.
- Five decision-time features only.
- Future CTR is not a model feature.
- Errors are inspected.
- Interpretation uses decision-support language, not causal language.
- No client names, domains, URLs, private queries, credentials, or raw warehouse exports are published.

**Run the notebook top-to-bottom with the private `HF_TOKEN` Colab Secret. The numerical results printed by the executed notebook are the authoritative results for the submission.**